In [2]:
import pandas as pd
import dask.dataframe as dd

In [3]:
tf = pd.read_csv("Trips_Full Data.csv")

In [4]:
td = pd.read_csv("Trips_by_Distance.csv")

In [5]:
td.columns = (td.columns.str.strip().str.lower().str.replace(" ","_"))
td = td.fillna(td.mean(numeric_only=True)) # fills missing numeric values with its mean 

In [6]:
# handles missing categorical values
td["state_postal_code"] = td["state_postal_code"].fillna("Unknown")
td["county_name"] = td["county_name"].fillna("Unknown")

In [7]:
tf.columns = (tf.columns.str.strip().str.lower().str.replace(" ","_"))  

In [38]:
from dask.distributed import Client
import time

# sequential processing
serial_start_time = time.time()

serial_group = td.groupby("week")["number_of_trips"].mean()

serial_time = time.time() - serial_start_time

print("Sequential processing time =", serial_time, "seconds")
print(serial_group)


# parallel processing
td_dask = dd.read_csv(
    "Trips_by_Distance.csv",
    dtype={
        "County Name": "object",
        "State Postal Code": "object",
        "Number of Trips": "float64",
        "Number of Trips <1": "float64",
        "Number of Trips 1-3": "float64",
        "Number of Trips 3-5": "float64",
        "Number of Trips 5-10": "float64",
        "Number of Trips 10-25": "float64",
        "Number of Trips 25-50": "float64",
        "Number of Trips 50-100": "float64",
        "Number of Trips 100-250": "float64",
        "Number of Trips 250-500": "float64",
        "Number of Trips >=500": "float64",
        "Population Staying at Home": "float64",
        "Population Not Staying at Home": "float64"
    }
)

td_dask.columns = td_dask.columns.str.strip().str.lower().str.replace(" ", "_")

processor_times = {}

for processors in [4, 8]:
    client = Client(n_workers=processors, threads_per_worker=1)

    parallel_start_time = time.time()

    parallel_group = td_dask.groupby("week")["number_of_trips"].mean().compute()

    parallel_time = time.time() - parallel_start_time
    processor_times[processors] = parallel_time

    print("Parallel time with", processors, "processors =", parallel_time, "seconds")
    print(parallel_group)

    client.close()


print("\nFinal comparison")
print("Sequential time =", serial_time, "seconds")
print("Parallel time with 4 processors =", processor_times[4], "seconds")
print("Parallel time with 8 processors =", processor_times[8], "seconds")

Sequential processing time = 0.19848895072937012 seconds
week
0     1.794149e+06
1     2.346483e+06
2     2.377830e+06
3     2.393895e+06
4     2.383902e+06
5     2.417372e+06
6     2.441275e+06
7     2.430313e+06
8     2.538681e+06
9     2.576819e+06
10    2.660709e+06
11    2.659115e+06
12    2.653295e+06
13    2.630632e+06
14    2.677912e+06
15    2.736895e+06
16    2.712637e+06
17    2.714436e+06
18    2.691101e+06
19    2.705980e+06
20    2.786707e+06
21    2.725621e+06
22    2.769102e+06
23    1.590359e+06
24    1.571096e+06
25    1.900221e+06
26    1.904758e+06
27    1.921256e+06
28    1.939462e+06
29    1.936240e+06
30    1.972308e+06
31    1.955124e+06
32    1.965232e+06
33    1.948573e+06
34    1.909640e+06
35    1.991166e+06
36    1.970724e+06
37    2.010845e+06
38    1.988273e+06
39    1.981355e+06
40    1.979193e+06
41    1.983318e+06
42    1.925476e+06
43    2.241266e+06
44    3.609361e+07
45    4.250206e+07
46    4.372210e+07
47    4.176082e+07
48    4.451167e+07
49    4

/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 40951 instead
  warnings.warn(
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/contextlib.py:126: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.09s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)
/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pand

Parallel time with 4 processors = 101.98992323875427 seconds
week
0     1.790985e+06
1     2.345837e+06
2     2.377387e+06
3     2.393534e+06
4     2.383482e+06
5     2.417149e+06
6     2.441151e+06
7     2.430129e+06
8     2.538922e+06
9     2.577167e+06
10    2.661305e+06
11    2.660380e+06
12    2.654859e+06
13    2.632218e+06
14    2.679961e+06
15    2.740260e+06
16    2.714534e+06
17    2.716004e+06
18    2.692524e+06
19    2.707423e+06
20    2.788673e+06
21    2.726937e+06
22    2.770649e+06
23    1.583955e+06
24    1.564553e+06
25    1.895085e+06
26    1.895194e+06
27    1.914513e+06
28    1.934126e+06
29    1.928896e+06
30    1.964468e+06
31    1.949685e+06
32    1.960152e+06
33    1.945032e+06
34    1.903549e+06
35    1.980865e+06
36    1.963274e+06
37    2.004892e+06
38    1.981126e+06
39    1.967660e+06
40    1.957318e+06
41    1.953017e+06
42    1.892724e+06
43    2.229870e+06
44    4.249873e+07
45    4.415141e+07
46    4.372210e+07
47    4.181478e+07
48    4.451167e+07
49 

2026-03-30 15:33:54,442 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-03-30 15:33:54,540 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-03-30 15:33:54,542 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-03-30 15:33:54,641 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42949 instead
  warnings.warn(
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/contextlib.py:126: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.20s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local 

Parallel time with 8 processors = 166.00766801834106 seconds
week
0     1.790985e+06
1     2.345837e+06
2     2.377387e+06
3     2.393534e+06
4     2.383482e+06
5     2.417149e+06
6     2.441151e+06
7     2.430129e+06
8     2.538922e+06
9     2.577167e+06
10    2.661305e+06
11    2.660380e+06
12    2.654859e+06
13    2.632218e+06
14    2.679961e+06
15    2.740260e+06
16    2.714534e+06
17    2.716004e+06
18    2.692524e+06
19    2.707423e+06
20    2.788673e+06
21    2.726937e+06
22    2.770649e+06
23    1.583955e+06
24    1.564553e+06
25    1.895085e+06
26    1.895194e+06
27    1.914513e+06
28    1.934126e+06
29    1.928896e+06
30    1.964468e+06
31    1.949685e+06
32    1.960152e+06
33    1.945032e+06
34    1.903549e+06
35    1.980865e+06
36    1.963274e+06
37    2.004892e+06
38    1.981126e+06
39    1.967660e+06
40    1.957318e+06
41    1.953017e+06
42    1.892724e+06
43    2.229870e+06
44    4.249873e+07
45    4.415141e+07
46    4.372210e+07
47    4.181478e+07
48    4.451167e+07
49 

2026-03-30 15:38:32,739 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/distributed/worker.py", line 1250, in heartbeat
    response = await retry_operation(
  File "/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/distributed/utils_comm.py", line 459, in retry_operation
    return await retry(
  File "/home/4b3f8652-ecd8-43c1-82e5-674b77dfb7d8/.local/lib/python3.9/site-packages/distributed/utils_comm.py", line 438, in retry
    return await coro()
  Fi

TimeoutError: 